In [1]:
import json
import os
from pathlib import Path
from pprint import pprint
import numpy as np

from datasets import load_from_disk
from transformers import AutoModelForTokenClassification, AutoTokenizer,\
DataCollatorForTokenClassification, TrainingArguments, Trainer
import evaluate

import wandb
from colorama import Fore, Style

In [2]:
# for type annotation
# print(type(tokenizer))
# print(type(model))
from transformers.models.bert.tokenization_bert import BertTokenizer
from transformers.models.distilbert.modeling_distilbert import DistilBertModel
from datasets import DatasetDict, Dataset

In [3]:
# model and tokenizer
model_path = "distilbert/distilbert-base-uncased"
tokenizer: BertTokenizer = AutoTokenizer.from_pretrained(model_path)

In [4]:
DATA_DIR = Path(os.getcwd()).parent / "data"
DATA_DIR.mkdir(exist_ok=True)
LABEL_INFO_PATH = DATA_DIR/"label_info.json"
DATASET_PATH = DATA_DIR/"cleaned_ai4privacy_300k_pii"

MODEL_DIR = Path(os.getcwd()).parent / "models"
MODEL_DIR.mkdir(exist_ok=True)
RUN_NAME = "pii_redaction_" + model_path.replace("/", "_") + "_v1"

In [5]:
dataset: DatasetDict = load_from_disk(DATASET_PATH) # type:ignore
print(f"dataset column names: {dataset["train"].column_names}\n\n")

dataset column names: ['source_text', 'privacy_mask']




In [6]:
# label info
with open(LABEL_INFO_PATH) as f:
    label_info: dict = json.load(f)
    
pprint(list(label_info.keys()))

['label2id', 'id2label', 'all_entities_counted', 'train_counted_entities']


In [7]:
label2id = label_info["label2id"]
id2label = label_info["id2label"]

In [8]:
example = dataset["train"][0]
print("dataset example:")
for item in example.items():
    print(item)

dataset example:
('source_text', 'Subject: Group Messaging for Admissions Process\n\nGood morning, everyone,\n\nI hope this message finds you well. As we continue our admissions processes, I would like to update you on the latest developments and key information. Please find below the timeline for our upcoming meetings:\n\n- wynqvrh053 - Meeting at 10:20am\n- luka.burg - Meeting at 21\n- qahil.wittauer - Meeting at quarter past 13\n- gholamhossein.ruschke - Meeting at 9:47 PM\n- pdmjrsyoz1460 ')
('privacy_mask', [{'value': 'wynqvrh053', 'start': 287, 'end': 297, 'label': 'USERNAME'}, {'value': '10:20am', 'start': 311, 'end': 318, 'label': 'TIME'}, {'value': 'luka.burg', 'start': 321, 'end': 330, 'label': 'USERNAME'}, {'value': '21', 'start': 344, 'end': 346, 'label': 'TIME'}, {'value': 'qahil.wittauer', 'start': 349, 'end': 363, 'label': 'USERNAME'}, {'value': 'quarter past 13', 'start': 377, 'end': 392, 'label': 'TIME'}, {'value': 'gholamhossein.ruschke', 'start': 395, 'end': 416, 'la

In [9]:
example_encodings = tokenizer(
    example["source_text"],
    return_offsets_mapping=True,
    add_special_tokens=False,
)
for item in example_encodings.items():
    print(item)

('input_ids', [3395, 1024, 2177, 24732, 2005, 20247, 2832, 2204, 2851, 1010, 3071, 1010, 1045, 3246, 2023, 4471, 4858, 2017, 2092, 1012, 2004, 2057, 3613, 2256, 20247, 6194, 1010, 1045, 2052, 2066, 2000, 10651, 2017, 2006, 1996, 6745, 8973, 1998, 3145, 2592, 1012, 3531, 2424, 2917, 1996, 17060, 2005, 2256, 9046, 6295, 1024, 1011, 1059, 6038, 4160, 19716, 2232, 2692, 22275, 1011, 3116, 2012, 2184, 1024, 2322, 3286, 1011, 11320, 2912, 1012, 20934, 10623, 1011, 3116, 2012, 2538, 1011, 1053, 4430, 4014, 1012, 15966, 2696, 13094, 1011, 3116, 2012, 4284, 2627, 2410, 1011, 1043, 14854, 3286, 15006, 20240, 2078, 1012, 22949, 2818, 3489, 1011, 3116, 2012, 1023, 1024, 4700, 7610, 1011, 22851, 2213, 3501, 2869, 7677, 2480, 16932, 16086])
('token_type_ids', [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [10]:
print(example_encodings.tokens())

['subject', ':', 'group', 'messaging', 'for', 'admissions', 'process', 'good', 'morning', ',', 'everyone', ',', 'i', 'hope', 'this', 'message', 'finds', 'you', 'well', '.', 'as', 'we', 'continue', 'our', 'admissions', 'processes', ',', 'i', 'would', 'like', 'to', 'update', 'you', 'on', 'the', 'latest', 'developments', 'and', 'key', 'information', '.', 'please', 'find', 'below', 'the', 'timeline', 'for', 'our', 'upcoming', 'meetings', ':', '-', 'w', '##yn', '##q', '##vr', '##h', '##0', '##53', '-', 'meeting', 'at', '10', ':', '20', '##am', '-', 'lu', '##ka', '.', 'bu', '##rg', '-', 'meeting', 'at', '21', '-', 'q', '##ah', '##il', '.', 'wit', '##ta', '##uer', '-', 'meeting', 'at', 'quarter', 'past', '13', '-', 'g', '##hol', '##am', '##hos', '##sei', '##n', '.', 'rus', '##ch', '##ke', '-', 'meeting', 'at', '9', ':', '47', 'pm', '-', 'pd', '##m', '##j', '##rs', '##yo', '##z', '##14', '##60']


In [11]:
for token, id_ in zip(example_encodings.tokens(), example_encodings.word_ids()):
    print(token, id_)

subject 0
: 1
group 2
messaging 3
for 4
admissions 5
process 6
good 7
morning 8
, 9
everyone 10
, 11
i 12
hope 13
this 14
message 15
finds 16
you 17
well 18
. 19
as 20
we 21
continue 22
our 23
admissions 24
processes 25
, 26
i 27
would 28
like 29
to 30
update 31
you 32
on 33
the 34
latest 35
developments 36
and 37
key 38
information 39
. 40
please 41
find 42
below 43
the 44
timeline 45
for 46
our 47
upcoming 48
meetings 49
: 50
- 51
w 52
##yn 52
##q 52
##vr 52
##h 52
##0 52
##53 52
- 53
meeting 54
at 55
10 56
: 57
20 58
##am 58
- 59
lu 60
##ka 60
. 61
bu 62
##rg 62
- 63
meeting 64
at 65
21 66
- 67
q 68
##ah 68
##il 68
. 69
wit 70
##ta 70
##uer 70
- 71
meeting 72
at 73
quarter 74
past 75
13 76
- 77
g 78
##hol 78
##am 78
##hos 78
##sei 78
##n 78
. 79
rus 80
##ch 80
##ke 80
- 81
meeting 82
at 83
9 84
: 85
47 86
pm 87
- 88
pd 89
##m 89
##j 89
##rs 89
##yo 89
##z 89
##14 89
##60 89


In [12]:
start, end = example_encodings.word_to_chars(52)
example["source_text"][start:end]

'wynqvrh053'

In [13]:
tokens = tokenizer.convert_ids_to_tokens(example_encodings["input_ids"])
print(tokens)

['subject', ':', 'group', 'messaging', 'for', 'admissions', 'process', 'good', 'morning', ',', 'everyone', ',', 'i', 'hope', 'this', 'message', 'finds', 'you', 'well', '.', 'as', 'we', 'continue', 'our', 'admissions', 'processes', ',', 'i', 'would', 'like', 'to', 'update', 'you', 'on', 'the', 'latest', 'developments', 'and', 'key', 'information', '.', 'please', 'find', 'below', 'the', 'timeline', 'for', 'our', 'upcoming', 'meetings', ':', '-', 'w', '##yn', '##q', '##vr', '##h', '##0', '##53', '-', 'meeting', 'at', '10', ':', '20', '##am', '-', 'lu', '##ka', '.', 'bu', '##rg', '-', 'meeting', 'at', '21', '-', 'q', '##ah', '##il', '.', 'wit', '##ta', '##uer', '-', 'meeting', 'at', 'quarter', 'past', '13', '-', 'g', '##hol', '##am', '##hos', '##sei', '##n', '.', 'rus', '##ch', '##ke', '-', 'meeting', 'at', '9', ':', '47', 'pm', '-', 'pd', '##m', '##j', '##rs', '##yo', '##z', '##14', '##60']


In [14]:
word_ids = example_encodings.word_ids()
print(word_ids)

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 52, 52, 52, 52, 52, 52, 53, 54, 55, 56, 57, 58, 58, 59, 60, 60, 61, 62, 62, 63, 64, 65, 66, 67, 68, 68, 68, 69, 70, 70, 70, 71, 72, 73, 74, 75, 76, 77, 78, 78, 78, 78, 78, 78, 79, 80, 80, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 89, 89, 89, 89, 89, 89, 89]


In [15]:
pprint(example["privacy_mask"], sort_dicts=False)

[{'value': 'wynqvrh053', 'start': 287, 'end': 297, 'label': 'USERNAME'},
 {'value': '10:20am', 'start': 311, 'end': 318, 'label': 'TIME'},
 {'value': 'luka.burg', 'start': 321, 'end': 330, 'label': 'USERNAME'},
 {'value': '21', 'start': 344, 'end': 346, 'label': 'TIME'},
 {'value': 'qahil.wittauer', 'start': 349, 'end': 363, 'label': 'USERNAME'},
 {'value': 'quarter past 13', 'start': 377, 'end': 392, 'label': 'TIME'},
 {'value': 'gholamhossein.ruschke',
  'start': 395,
  'end': 416,
  'label': 'USERNAME'},
 {'value': '9:47 PM', 'start': 430, 'end': 437, 'label': 'TIME'},
 {'value': 'pdmjrsyoz1460', 'start': 440, 'end': 453, 'label': 'USERNAME'}]


In [16]:
token_offsets = tokenizer(
        example["source_text"],
        return_offsets_mapping=True,
        add_special_tokens=False,
    )["offset_mapping"]
print(token_offsets)
print(len(token_offsets))

[(0, 7), (7, 8), (9, 14), (15, 24), (25, 28), (29, 39), (40, 47), (49, 53), (54, 61), (61, 62), (63, 71), (71, 72), (74, 75), (76, 80), (81, 85), (86, 93), (94, 99), (100, 103), (104, 108), (108, 109), (110, 112), (113, 115), (116, 124), (125, 128), (129, 139), (140, 149), (149, 150), (151, 152), (153, 158), (159, 163), (164, 166), (167, 173), (174, 177), (178, 180), (181, 184), (185, 191), (192, 204), (205, 208), (209, 212), (213, 224), (224, 225), (226, 232), (233, 237), (238, 243), (244, 247), (248, 256), (257, 260), (261, 264), (265, 273), (274, 282), (282, 283), (285, 286), (287, 288), (288, 290), (290, 291), (291, 293), (293, 294), (294, 295), (295, 297), (298, 299), (300, 307), (308, 310), (311, 313), (313, 314), (314, 316), (316, 318), (319, 320), (321, 323), (323, 325), (325, 326), (326, 328), (328, 330), (331, 332), (333, 340), (341, 343), (344, 346), (347, 348), (349, 350), (350, 352), (352, 354), (354, 355), (355, 358), (358, 360), (360, 363), (364, 365), (366, 373), (374, 

In [17]:
def get_ner_labels(batch: dict[str, list]) -> dict[str, list[list[str]]]:
    token_offsets = tokenizer(
        batch["source_text"],
        return_offsets_mapping=True,
        add_special_tokens=False,
    )["offset_mapping"]

    batch_ner_labels: list[list[str]] = []
    # loop through the batch to get the privacy masks for every sequence
    for i, row_masks in enumerate(batch["privacy_mask"]):
        row_ner_labels = []
        # loop through the token_offsets of the current sequence
        for offset in token_offsets[i]:
            # if add_special_tokens is true skip over special tokens (offset with (0,0))
            if offset == (0, 0): 
                row_ner_labels.append("O")
                continue
            # label is "O" unless privacy label is found
            label = "O" 
            # loop through masks to find label
            for privacy_mask in row_masks:
                # end tag is exclusive (just like python indexing)
                if offset[0] >= privacy_mask["start"] and offset[1] <= privacy_mask["end"]:
                    label = privacy_mask["label"]
                    if offset[0] == privacy_mask["start"]:
                        label = "B-" + label
                    else:
                        label = "I-" + label
                    
                    break # break out of for loop when/if label is found
            row_ner_labels.append(label)
            
        batch_ner_labels.append(row_ner_labels)
    
    return {"ner_tags": batch_ner_labels}

In [18]:
dataset = dataset.map(get_ner_labels, batched=True, batch_size=20_000)
pprint(dataset["train"].features)

{'ner_tags': List(Value('string')),
 'privacy_mask': List({'value': Value('string'), 'start': Value('int64'), 'end': Value('int64'), 'label': Value('string')}),
 'source_text': Value('string')}


In [19]:
tagged_example = dataset["train"][0]
tags = tagged_example["ner_tags"]
print(tags)
print(len(tags))

['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-USERNAME', 'I-USERNAME', 'I-USERNAME', 'I-USERNAME', 'I-USERNAME', 'I-USERNAME', 'I-USERNAME', 'O', 'O', 'O', 'B-TIME', 'I-TIME', 'I-TIME', 'I-TIME', 'O', 'B-USERNAME', 'I-USERNAME', 'I-USERNAME', 'I-USERNAME', 'I-USERNAME', 'O', 'O', 'O', 'B-TIME', 'O', 'B-USERNAME', 'I-USERNAME', 'I-USERNAME', 'I-USERNAME', 'I-USERNAME', 'I-USERNAME', 'I-USERNAME', 'O', 'O', 'O', 'B-TIME', 'I-TIME', 'I-TIME', 'O', 'B-USERNAME', 'I-USERNAME', 'I-USERNAME', 'I-USERNAME', 'I-USERNAME', 'I-USERNAME', 'I-USERNAME', 'I-USERNAME', 'I-USERNAME', 'I-USERNAME', 'O', 'O', 'O', 'B-TIME', 'I-TIME', 'I-TIME', 'I-TIME', 'O', 'B-USERNAME', 'I-USERNAME', 'I-USERNAME', 'I-USERNAME', 'I-USERNAME', 'I-USERNAME', 'I-USERNAME', 'I-USERNAME']
117


In [20]:
for tag, token, word_id in zip(tags, example_encodings.tokens(), example_encodings.word_ids()):
    if tag != "O":
        print(f"{token=}, {tag=}, {word_id=}")

token='w', tag='B-USERNAME', word_id=52
token='##yn', tag='I-USERNAME', word_id=52
token='##q', tag='I-USERNAME', word_id=52
token='##vr', tag='I-USERNAME', word_id=52
token='##h', tag='I-USERNAME', word_id=52
token='##0', tag='I-USERNAME', word_id=52
token='##53', tag='I-USERNAME', word_id=52
token='10', tag='B-TIME', word_id=56
token=':', tag='I-TIME', word_id=57
token='20', tag='I-TIME', word_id=58
token='##am', tag='I-TIME', word_id=58
token='lu', tag='B-USERNAME', word_id=60
token='##ka', tag='I-USERNAME', word_id=60
token='.', tag='I-USERNAME', word_id=61
token='bu', tag='I-USERNAME', word_id=62
token='##rg', tag='I-USERNAME', word_id=62
token='21', tag='B-TIME', word_id=66
token='q', tag='B-USERNAME', word_id=68
token='##ah', tag='I-USERNAME', word_id=68
token='##il', tag='I-USERNAME', word_id=68
token='.', tag='I-USERNAME', word_id=69
token='wit', tag='I-USERNAME', word_id=70
token='##ta', tag='I-USERNAME', word_id=70
token='##uer', tag='I-USERNAME', word_id=70
token='quarter',

In [21]:
def tokenize_and_align_labels_single(example: dict[str, list]):
    tokenized_input = tokenizer(
        example["source_text"],
        truncation=True,
        add_special_tokens=False
    )
    labels = []
    word_ids = tokenized_input.word_ids()
    for i, tag in enumerate(example["ner_tags"]):
        previous_word_idx = None

        word_idx = word_ids[i]
        
        if word_idx is None:
            labels.append(-100)
        elif word_idx != previous_word_idx:
            # in case there is a label not in label2id in becomes 0
            labels.append(label2id.get(tag, 0))
        else:
            labels.append(-100)
        previous_word_idx = word_idx
            
    tokenized_input["labels"] = labels
    return tokenized_input

In [22]:
def tokenize_and_align_labels(batch: dict[str, list[list]]):
    tokenized_inputs = tokenizer(
        batch["source_text"],
        truncation=True,
        add_special_tokens=False
    )
    batch_labels = []
    for row_idx, row_tags in enumerate(batch["ner_tags"]):
        row_labels = []
        row_word_id = tokenized_inputs.word_ids(batch_index=row_idx)
        prev_word_id = None
        for word_id, tag in zip(row_word_id, row_tags):
            if word_id is None:
                row_labels.append(-100)
            elif word_id != prev_word_id:
                row_labels.append(label2id.get(tag, 0))
            else:
                row_labels.append(-100)
            prev_word_id = word_id
        batch_labels.append(row_labels)
    tokenized_inputs["labels"] = batch_labels
    return tokenized_inputs
            

In [23]:
# def tokenize_and_align_labels(batch: dict[str, list[list]]):
#     tokenized_inputs = tokenizer(
#         batch["source_text"],
#         truncation=True,
#         add_special_tokens=False
#     )
#     batch_labels = []
#     for i, tags in enumerate(batch["ner_tags"]):
#         word_ids = tokenized_inputs.word_ids(batch_index=i)
#         previous_word_idx = None
#         row_labels = []
#         for word_idx, tag in zip(word_ids, tags):
#             if word_idx is None:
#                 row_labels.append(-100)
#             elif word_idx != previous_word_idx:
#                 # in case there is a label not in label2id in becomes 0
#                 row_labels.append(label2id.get(tag, 0))
#             else:
#                 row_labels.append(-100)
                
#             previous_word_idx = word_idx
#         batch_labels.append(row_labels)
#     tokenized_inputs["labels"] = batch_labels
#     return tokenized_inputs

In [24]:
labeled_example = tokenize_and_align_labels_single(tagged_example)
print(labeled_example["labels"])

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 2, 2, 2, 2, 2, 0, 0, 0, 3, 4, 4, 4, 0, 1, 2, 2, 2, 2, 0, 0, 0, 3, 0, 1, 2, 2, 2, 2, 2, 2, 0, 0, 0, 3, 4, 4, 0, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 0, 0, 0, 3, 4, 4, 4, 0, 1, 2, 2, 2, 2, 2, 2, 2]


In [25]:
for tag, label in zip(tags, labeled_example["labels"]):
    print(f"{tag=}, {label=}")

tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='B-USERNAME', label=1
tag='I-USERNAME', label=2
tag='I-USERNAME', label=2
tag='I-USERNAME', label=2
tag='I-USERN

In [26]:
ex_wi = tokenizer(
    tagged_example["source_text"],
    truncation=True,
    add_special_tokens=False
).word_ids()
masks = tagged_example["privacy_mask"]

token_offsets = tokenizer(
    tagged_example["source_text"],
    return_offsets_mapping=True,
    add_special_tokens=False,
)["offset_mapping"]

id2label = {i: label for label, i in label2id.items()}
tokens = tokenizer.convert_ids_to_tokens(labeled_example["input_ids"])

In [27]:
for mask in masks:
    print(Fore.BLUE + f"{mask}" + Style.RESET_ALL)    
for label, wi, offset in zip(labeled_example["labels"], ex_wi, token_offsets):
    if label != -100 and label != 0:
        token = tokens[wi] #type:ignore
        value = f"label={id2label[label]}, word_id={wi}, {token=}, {offset=}"
        print(value)
        
        masks_str = []
        for mask in masks:
            if offset[0] >= mask["start"] and offset[1] <= mask["end"]:
                masks_str.append(f"in {mask}")
                
        if masks_str:
            print(Fore.GREEN + f"{masks_str}")
            print(Style.RESET_ALL + "", end="")
        else:
            print(Fore.RED + f" {offset} not in masks" )
            print(Style.RESET_ALL + "", end="")
        
    # elif label == -100 and wi != None:
    #     token = tokens[wi]
    #     print(f"label = {label} word_id = {wi}, token = {token}")

{'value': 'wynqvrh053', 'start': 287, 'end': 297, 'label': 'USERNAME'}
{'value': '10:20am', 'start': 311, 'end': 318, 'label': 'TIME'}
{'value': 'luka.burg', 'start': 321, 'end': 330, 'label': 'USERNAME'}
{'value': '21', 'start': 344, 'end': 346, 'label': 'TIME'}
{'value': 'qahil.wittauer', 'start': 349, 'end': 363, 'label': 'USERNAME'}
{'value': 'quarter past 13', 'start': 377, 'end': 392, 'label': 'TIME'}
{'value': 'gholamhossein.ruschke', 'start': 395, 'end': 416, 'label': 'USERNAME'}
{'value': '9:47 PM', 'start': 430, 'end': 437, 'label': 'TIME'}
{'value': 'pdmjrsyoz1460', 'start': 440, 'end': 453, 'label': 'USERNAME'}
label=B-USERNAME, word_id=52, token='w', offset=(287, 288)
["in {'value': 'wynqvrh053', 'start': 287, 'end': 297, 'label': 'USERNAME'}"]
label=I-USERNAME, word_id=52, token='w', offset=(288, 290)
["in {'value': 'wynqvrh053', 'start': 287, 'end': 297, 'label': 'USERNAME'}"]
label=I-USERNAME, word_id=52, token='w', offset=(290, 291)
["in {'value': 'wynqvrh053', 'start'

In [28]:
example = dataset["train"][0]
example_encodings = tokenizer(
    example["source_text"],
    return_offsets_mapping=True,
    add_special_tokens=False,
)
line1 = ""
line2 = ""
for word, label in zip(example_encodings.tokens(), example["ner_tags"]):
    full_label = label
    max_length = max(len(word), len(full_label))
    line1 += word + " " * (max_length - len(word) + 1)
    line2 += full_label + " " * (max_length - len(full_label) + 1)

print(line1)
print(line2)

subject : group messaging for admissions process good morning , everyone , i hope this message finds you well . as we continue our admissions processes , i would like to update you on the latest developments and key information . please find below the timeline for our upcoming meetings : - w          ##yn       ##q        ##vr       ##h        ##0        ##53       - meeting at 10     :      20     ##am   - lu         ##ka       .          bu         ##rg       - meeting at 21     - q          ##ah       ##il       .          wit        ##ta       ##uer      - meeting at quarter past   13     - g          ##hol      ##am       ##hos      ##sei      ##n        .          rus        ##ch       ##ke       - meeting at 9      :      47     pm     - pd         ##m        ##j        ##rs       ##yo       ##z        ##14       ##60       
O       O O     O         O   O          O       O    O       O O        O O O    O    O       O     O   O    O O  O  O        O   O          O         O O 

In [29]:
dataset = dataset.map(tokenize_and_align_labels, batched=True, batch_size=20_000)

In [30]:
labeled_example = dataset["train"][0]
print(labeled_example["labels"])

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, -100, -100, -100, -100, -100, -100, 0, 0, 0, 3, 4, 4, -100, 0, 1, -100, 2, 2, -100, 0, 0, 0, 3, 0, 1, -100, -100, 2, 2, -100, -100, 0, 0, 0, 3, 4, 4, 0, 1, -100, -100, -100, -100, -100, 2, 2, -100, -100, 0, 0, 0, 3, 4, 4, 4, 0, 1, -100, -100, -100, -100, -100, -100, -100]


In [31]:
for tag, label in zip(tags, labeled_example["labels"]):
    print(f"{tag=}, {label=}")

tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='O', label=0
tag='B-USERNAME', label=1
tag='I-USERNAME', label=-100
tag='I-USERNAME', label=-100
tag='I-USERNAME', label=-100
tag

In [32]:
pprint(dataset["train"].features)

{'attention_mask': List(Value('int8')),
 'input_ids': List(Value('int32')),
 'labels': List(Value('int64')),
 'ner_tags': List(Value('string')),
 'privacy_mask': List({'value': Value('string'), 'start': Value('int64'), 'end': Value('int64'), 'label': Value('string')}),
 'source_text': Value('string'),
 'token_type_ids': List(Value('int8'))}


In [33]:
for key, value in dataset["train"][0].items():
    print(f"{key}:{len(value)}")
    
print(token_offsets[0], token_offsets[-1])

source_text:454
privacy_mask:9
ner_tags:117
input_ids:117
token_type_ids:117
attention_mask:117
labels:117
(0, 7) (451, 453)


In [34]:
example = dataset["train"][0]
ex_wi = tokenizer(
    example["source_text"],
    truncation=True,
    add_special_tokens=False
).word_ids(batch_index=0)
masks = example["privacy_mask"]

token_offsets = tokenizer(
    example["source_text"],
    return_offsets_mapping=True,
    add_special_tokens=False,
)["offset_mapping"]

id2label = {i: label for label, i in label2id.items()}
tokens = tokenizer.convert_ids_to_tokens(example["input_ids"])

In [35]:
print(len(example["labels"]))
print(len(tokens))

117
117


In [36]:
for mask in masks:
    print(Fore.BLUE + f"{mask}" + Style.RESET_ALL)    
for label, wi, offset in zip(example["labels"], ex_wi, token_offsets):
    if label != -100 and label != 0:
        token = tokens[wi] #type:ignore
        value = f"label={id2label[label]}, word_id={wi}, {token=}, {offset=}"
        print(value)
        
        masks_str = []
        for mask in masks:
            if offset[0] >= mask["start"] and offset[1] <= mask["end"]:
                masks_str.append(f"in {mask}")
                
        if masks_str:
            print(Fore.GREEN + f"{masks_str}")
            print(Style.RESET_ALL + "", end="")
        else:
            print(Fore.RED + f" {offset} not in masks" )
            print(Style.RESET_ALL + "", end="")
        
    # elif label == -100 and wi != None:
    #     token = tokens[wi]
    #     print(f"label = {label} word_id = {wi}, token = {token}")

{'value': 'wynqvrh053', 'start': 287, 'end': 297, 'label': 'USERNAME'}
{'value': '10:20am', 'start': 311, 'end': 318, 'label': 'TIME'}
{'value': 'luka.burg', 'start': 321, 'end': 330, 'label': 'USERNAME'}
{'value': '21', 'start': 344, 'end': 346, 'label': 'TIME'}
{'value': 'qahil.wittauer', 'start': 349, 'end': 363, 'label': 'USERNAME'}
{'value': 'quarter past 13', 'start': 377, 'end': 392, 'label': 'TIME'}
{'value': 'gholamhossein.ruschke', 'start': 395, 'end': 416, 'label': 'USERNAME'}
{'value': '9:47 PM', 'start': 430, 'end': 437, 'label': 'TIME'}
{'value': 'pdmjrsyoz1460', 'start': 440, 'end': 453, 'label': 'USERNAME'}
label=B-USERNAME, word_id=52, token='w', offset=(287, 288)
["in {'value': 'wynqvrh053', 'start': 287, 'end': 297, 'label': 'USERNAME'}"]
label=B-TIME, word_id=56, token='##h', offset=(311, 313)
["in {'value': '10:20am', 'start': 311, 'end': 318, 'label': 'TIME'}"]
label=I-TIME, word_id=57, token='##0', offset=(313, 314)
["in {'value': '10:20am', 'start': 311, 'end': 

In [37]:
example = dataset["train"][0]
example_encodings = tokenizer(
    example["source_text"],
    return_offsets_mapping=True,
    add_special_tokens=False,
)
line1 = ""
line2 = ""
for word, label in zip(example_encodings.tokens(), example["ner_tags"]):
    full_label = label
    max_length = max(len(word), len(full_label))
    line1 += word + " " * (max_length - len(word) + 1)
    line2 += full_label + " " * (max_length - len(full_label) + 1)

print(line1)
print(line2)

subject : group messaging for admissions process good morning , everyone , i hope this message finds you well . as we continue our admissions processes , i would like to update you on the latest developments and key information . please find below the timeline for our upcoming meetings : - w          ##yn       ##q        ##vr       ##h        ##0        ##53       - meeting at 10     :      20     ##am   - lu         ##ka       .          bu         ##rg       - meeting at 21     - q          ##ah       ##il       .          wit        ##ta       ##uer      - meeting at quarter past   13     - g          ##hol      ##am       ##hos      ##sei      ##n        .          rus        ##ch       ##ke       - meeting at 9      :      47     pm     - pd         ##m        ##j        ##rs       ##yo       ##z        ##14       ##60       
O       O O     O         O   O          O       O    O       O O        O O O    O    O       O     O   O    O O  O  O        O   O          O         O O 

In [38]:
print(f"{len(example["ner_tags"])=}")
print(f"{len(example["input_ids"])=}")

len(example["ner_tags"])=117
len(example["input_ids"])=117


In [39]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

seqeval = evaluate.load("seqeval")

In [40]:
model:DistilBertModel = AutoModelForTokenClassification.from_pretrained(
    model_path, num_labels=len(id2label), id2label=id2label, label2id=label2id)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForTokenClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [41]:
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/zelluzy/.netrc.
wandb: Currently logged in as: bengid (bengid-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [42]:
from transformers import EvalPrediction
def compute_metrics(p: EvalPrediction) -> dict[str, float]:
    predictions, label_ids = p.predictions, p.label_ids
    predictions = np.argmax(predictions, axis=-1)
    
    true_preds = [
        [id2label[pred] for pred, tgt_id in zip(preds_row, labels_row) if tgt_id != -100]
        for preds_row, labels_row in zip(predictions, label_ids)
    ]
    
    true_labels =[
        [id2label[tgt_id] for tgt_id in labels_row if tgt_id != -100]
        for labels_row in label_ids
    ]
    
    results = seqeval.compute(predictions=true_preds, references=true_labels)
    
    flat = {}
    if results is not None:
        flat.update({
            "precision": results["overall_precision"],
            "recall": results["overall_recall"],
            "f1": results["overall_f1"],
            "accuracy": results["overall_accuracy"],
        })
        
        for entity, scores in results.items():
            if isinstance(scores, dict): # filter out scores for individual labels
                flat[f"{entity}_f1"] = scores["f1"]
                flat[f"{entity}_support"] = scores["number"]
    
    return flat

In [43]:
training_args = TrainingArguments(
    output_dir=str(MODEL_DIR),
    report_to="wandb",
    run_name=RUN_NAME,
    learning_rate=2e-5, # low learning_rate
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    max_grad_norm=1.0, # default
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
wandb.init(project="pii-redaction")
trainer_output = trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy,Bod F1,Bod Support,Building F1,Building Support,City F1,City Support,Country F1,Country Support,Date F1,Date Support,Driverlicense F1,Driverlicense Support,Email F1,Email Support,Geocoord F1,Geocoord Support,Givenname1 F1,Givenname1 Support,Givenname2 F1,Givenname2 Support,Idcard F1,Idcard Support,Ip F1,Ip Support,Lastname1 F1,Lastname1 Support,Lastname2 F1,Lastname2 Support,Lastname3 F1,Lastname3 Support,Pass F1,Pass Support,Passport F1,Passport Support,Postcode F1,Postcode Support,Secaddress F1,Secaddress Support,Sex F1,Sex Support,Socialnumber F1,Socialnumber Support,State F1,State Support,Street F1,Street Support,Tel F1,Tel Support,Time F1,Time Support,Title F1,Title Support,Username F1,Username Support
1,0.092803,0.053930,0.883380,0.895028,0.889166,0.986840,0.951682,1148,0.968358,987,0.926899,1005,0.947849,765,0.891750,838,0.881674,1201,0.967219,1272,0.850242,104,0.654051,954,0.165017,268,0.891313,1352,0.965288,1046,0.655855,1212,0.266409,318,0.000000,106,0.856102,804,0.884120,1214,0.957370,959,0.960272,442,0.952864,980,0.912361,1316,0.952151,1020,0.934284,971,0.930889,1037,0.954617,1876,0.927857,957,0.885703,1331
2,0.054732,0.047483,0.904242,0.920104,0.912104,0.988084,0.947011,1148,0.972727,987,0.947317,1005,0.958146,765,0.898780,838,0.912829,1201,0.975175,1272,0.932692,104,0.758998,954,0.576271,268,0.910694,1352,0.960000,1046,0.725384,1212,0.503477,318,0.000000,106,0.883466,804,0.903481,1214,0.970833,959,0.964813,442,0.958669,980,0.936748,1316,0.974209,1020,0.945823,971,0.945780,1037,0.954283,1876,0.946758,957,0.914569,1331
3,0.044240,0.043665,0.907663,0.929639,0.918520,0.988932,0.960904,1148,0.976791,987,0.950563,1005,0.953885,765,0.908246,838,0.928938,1201,0.975039,1272,0.942308,104,0.787067,954,0.626058,268,0.912698,1352,0.978541,1046,0.734797,1212,0.502941,318,0.018519,106,0.869461,804,0.923200,1214,0.970664,959,0.968254,442,0.963038,980,0.943323,1316,0.978474,1020,0.948433,971,0.940682,1037,0.958997,1876,0.957120,957,0.913852,1331


/home/zelluzy/Desktop/pii_redaction_api/.venv/lib/python3.14/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/home/zelluzy/Desktop/pii_redaction_api/.venv/lib/python3.14/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
def log_predictions_table(trainer: Trainer, test_ds: Dataset | DatasetDict, 
                          tokenizer: BertTokenizer, id2label: dict, 
                          n_samples:int=50) -> wandb.Table:
    predictions, labels, _ = trainer.predict(test_ds) # type:ignore
    preds = np.argmax(predictions, axis=2)
    
    table = wandb.Table(columns=["sequence", "true_labels", "pred_labels", "accuracy"])
    
    for i in range(min(len(test_ds), n_samples)):
        tokens = tokenizer.convert_ids_to_tokens(test_ds[i]["input_ids"])
        
        row_tokens, true_row, pred_row = [], [], []
        for token, true_id, pred_id in zip(tokens, labels[i], preds[i]):
            if true_id == -100:
                continue
            row_tokens.append(token)
            true_row.append(id2label[true_id])
            pred_row.append(id2label[pred_id])

        correct = np.where(np.array(true_row)==np.array(pred_row), 1, 0) 
        accuracy = float(np.sum(correct) / len(correct)) * 100
        table.add_data(
            " ".join(row_tokens),
            " ".join(true_row),
            " ".join(pred_row),
            f"{accuracy:.2f}%"
        )
        
    wandb.log({"predictions": table})
    return table
            

In [ ]:
table = log_predictions_table(trainer, dataset["test"], tokenizer, id2label)

In [ ]:
import pandas as pd
table_df = pd.DataFrame(table.data, columns=table.columns)
display(table_df.head)

<bound method NDFrame.head of                                              sequence  \
0   * * guest injury report * * * * date : * * 14 ...   
1   ions for improvement . dear 1947 , i would lik...   
2   " ec enrollment form child information : child...   
3   lear - * * sex : * * masculine - * * email : *...   
4   143 - time of access : 6 pm 8 . driver h : - d...   
5   tt : 3 : 19 # # # # part 3 - reason for exempt...   
6   d eye contact and engaged actively in therapy ...   
7   { " patient _ certification " : { " 49 : a : 4...   
8   subject : innovative conflict resolution strat...   
9   < access > 14 : 44 < / access > < / security >...   
10  { " project title " : " empowerment through ed...   
11  ill " , " city " : " lew " , " state " : " eng...   
12  last | comment 1 | p @ ao . com | 310 - 39 - 6...   
13  { " sec " : { " authorization " : { " authoriz...   
14  " engage in active listening and valid concern...   
15  { " observation _ records " : [ { " patient _ ...   
1

In [ ]:
table_df.head

<bound method NDFrame.head of                                              sequence  \
0   * * guest injury report * * * * date : * * 14 ...   
1   ions for improvement . dear 1947 , i would lik...   
2   " ec enrollment form child information : child...   
3   lear - * * sex : * * masculine - * * email : *...   
4   143 - time of access : 6 pm 8 . driver h : - d...   
5   tt : 3 : 19 # # # # part 3 - reason for exempt...   
6   d eye contact and engaged actively in therapy ...   
7   { " patient _ certification " : { " 49 : a : 4...   
8   subject : innovative conflict resolution strat...   
9   < access > 14 : 44 < / access > < / security >...   
10  { " project title " : " empowerment through ed...   
11  ill " , " city " : " lew " , " state " : " eng...   
12  last | comment 1 | p @ ao . com | 310 - 39 - 6...   
13  { " sec " : { " authorization " : { " authoriz...   
14  " engage in active listening and valid concern...   
15  { " observation _ records " : [ { " patient _ ...   
1

In [ ]:
trainer_output

TrainOutput(global_step=9350, training_loss=0.3364608062907336, metrics={'train_runtime': 659.1712, 'train_samples_per_second': 226.861, 'train_steps_per_second': 14.184, 'total_flos': 7469725779630432.0, 'train_loss': 0.3364608062907336, 'epoch': 5.0})

In [ ]:
dataset["test"][0]

{'source_text': '**Guest Injury Report**\n\n**Date:** 14/11/2008\n\n**Time:** 17:52:39\n\n**Location:** 684\n\n---\n\n**Guest 1 Details:**\n- **Title:** Duchess\n- **Name:** Puspita Langenick\n- **IP Address:** e00e:f623:c39e:5a15:3da9:c968:c58c:e325\n- **Password:** m7Fk7/J&\\\\\n- **Check-in Time:** 8 PM\n\n---\n\n**Guest 2 Details:**\n- **Title:** Arch',
 'privacy_mask': [{'value': '14/11/2008',
   'start': 35,
   'end': 45,
   'label': 'DATE'},
  {'value': '17:52:39', 'start': 57, 'end': 65, 'label': 'TIME'},
  {'value': '684', 'start': 81, 'end': 84, 'label': 'BUILDING'},
  {'value': 'Duchess', 'start': 125, 'end': 132, 'label': 'TITLE'},
  {'value': 'Puspita', 'start': 145, 'end': 152, 'label': 'GIVENNAME1'},
  {'value': 'Langenick', 'start': 153, 'end': 162, 'label': 'LASTNAME1'},
  {'value': 'e00e:f623:c39e:5a15:3da9:c968:c58c:e325',
   'start': 181,
   'end': 220,
   'label': 'IP'},
  {'value': 'm7Fk7/J&\\\\', 'start': 237, 'end': 247, 'label': 'PASS'},
  {'value': '8 PM', 'st

In [ ]:
# from transformers.integrations import WandbCallback

# trainer.remove_callback(WandbCallback)
out = trainer.predict(dataset["test"])
preds = np.argmax(out.predictions, axis=-1)
print(preds)
print(Fore.CYAN + "")
pprint(out.metrics)

/home/zelluzy/Desktop/pii_redaction_api/.venv/lib/python3.14/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]

{'test_BOD_f1': 0.6021400778210118,
 'test_BOD_support': 1045,
 'test_BUILDING_f1': 0.36727879799666113,
 'test_BUILDING_support': 430,
 'test_CITY_f1': 0.40804020100502514,
 'test_CITY_support': 607,
 'test_COUNTRY_f1': 0.4129353233830846,
 'test_COUNTRY_support': 287,
 'test_DATE_f1': 0.588579795021962,
 'test_DATE_support': 668,
 'test_DRIVERLICENSE_f1': 0.3601453035806954,
 'test_DRIVERLICENSE_support': 984,
 'test_EMAIL_f1': 0.4917337855023315,
 'test_EMAIL_support': 1161,
 'test_GEOCOORD_f1': 0.4171122994652406,
 'test_GEOCOORD_support': 89,
 'test_GIVENNAME1_f1': 0.2899454403741232,
 'test_GIVENNAME1_support': 765,
 'test_GIVENNAME2_f1': 0.0,
 'test_GIVENNAME2_support': 213,
 'test_IDCARD_f1': 0.4195672624647224,
 'test_IDCARD_support': 1034,
 'test_IP_f1': 0.3760416666666666,
 'test_IP_support': 944,
 'test_LASTNAME1_f1': 0.27365728900255754,
 'test_LASTNAME1

In [ ]:
threshold = 0.5
missed = {}
passed = {}
for title, score in out.metrics.items():
    if score < threshold:
        missed[title] = score
    elif not title.endswith("support"):
        passed[title] = score

pprint(missed)

{'test_BUILDING_f1': 0.36727879799666113,
 'test_CITY_f1': 0.40804020100502514,
 'test_COUNTRY_f1': 0.4129353233830846,
 'test_DRIVERLICENSE_f1': 0.3601453035806954,
 'test_EMAIL_f1': 0.4917337855023315,
 'test_GEOCOORD_f1': 0.4171122994652406,
 'test_GIVENNAME1_f1': 0.2899454403741232,
 'test_GIVENNAME2_f1': 0.0,
 'test_IDCARD_f1': 0.4195672624647224,
 'test_IP_f1': 0.3760416666666666,
 'test_LASTNAME1_f1': 0.27365728900255754,
 'test_LASTNAME2_f1': 0.008064516129032258,
 'test_LASTNAME3_f1': 0.0,
 'test_PASSPORT_f1': 0.41308793456032716,
 'test_PASS_f1': 0.25998433829287393,
 'test_POSTCODE_f1': 0.4572713643178411,
 'test_SECADDRESS_f1': 0.3838771593090211,
 'test_SEX_f1': 0.46408839779005523,
 'test_STATE_f1': 0.0,
 'test_STREET_f1': 0.3623747108712413,
 'test_TEL_f1': 0.44148319814600234,
 'test_TITLE_f1': 0.45323741007194246,
 'test_USERNAME_f1': 0.43854119765871225,
 'test_f1': 0.4238443121124685,
 'test_loss': 0.1913127303123474,
 'test_precision': 0.45922575495543433,
 'test_re

In [ ]:
pprint(passed)

{'test_BOD_f1': 0.6021400778210118,
 'test_DATE_f1': 0.588579795021962,
 'test_SOCIALNUMBER_f1': 0.5057142857142857,
 'test_TIME_f1': 0.5222319093286836,
 'test_accuracy': 0.9372172402649863,
 'test_runtime': 8.9266,
 'test_samples_per_second': 445.073,
 'test_steps_per_second': 27.894}


In [40]:
wandb.finish()

eval/BOD_f1,▁▅▆██
eval/BOD_support,▁▁▁▁▁
eval/BUILDING_f1,▁▆▇▇█
eval/BUILDING_support,▁▁▁▁▁
eval/CITY_f1,▁▄▆▇█
eval/CITY_support,▁▁▁▁▁
eval/COUNTRY_f1,▁▅▇▇█
eval/COUNTRY_support,▁▁▁▁▁
eval/DATE_f1,▁▄▆██
eval/DATE_support,▁▁▁▁▁
+119,...
